In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Configuración para encontrar la raí­z del proyecto y agregarla al sys.path
# Necesario para importar módulos desde 'src'
cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if project_root is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from src.geospatial_expansion import agregar_distancias_minimas_poi

In [3]:
DATA_DIR = Path("../../data/raw/idealistaAPI/preprocess/sale_homes_run_20260331_174125")
DATA_IDEALISTA = DATA_DIR / "sale_homes_cantabria_bezana_like_raw.csv"

df = pd.read_csv(DATA_IDEALISTA)

df.head()

,propertyCode,thumbnail,externalReference,numPhotos,price,propertyType,operation,size,rooms,bathrooms,address,province,municipality,district,country,latitude,longitude,showAddress,url,distance,description,hasVideo,status,newDevelopment,priceByArea,hasPlan,has3DTour,has360,hasStaging,notes,topNewDevelopment,newDevelopmentHighlight,topPlus,priceInfo.price.amount,priceInfo.price.currencySuffix,parkingSpace.hasParkingSpace,parkingSpace.isParkingSpaceIncludedInPrice,detailedType.typology,detailedType.subTypology,suggestedTexts.subtitle,suggestedTexts.title,highlight.groupDescription,floor,exterior,hasLift,priceInfo.price.priceDropInfo.formerPrice,priceInfo.price.priceDropInfo.priceDropValue,priceInfo.price.priceDropInfo.priceDropPercentage,parkingSpace.parkingSpacePrice,newDevelopmentFinished
0,111067785,https://img4.idealista.com/blur/480_360_mq/0/i...,MV420,23,600000.0,chalet,sale,216.0,5,3,Urbanizacion los Alamos I,Cantabria,Santa Cruz de Bezana,Bezana,es,43.445929,-3.904541,False,https://www.idealista.com/inmueble/111067785/,5336,Mena Inmobiliaria vende magnífico chalet parea...,False,good,False,2778.0,False,False,False,False,[],False,False,False,600000.0,€,True,True,chalet,semidetachedHouse,"Bezana, Santa Cruz de Bezana",Chalet pareado en Urbanizacion los Alamos I,Destacado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,111067361,https://img4.idealista.com/blur/480_360_mq/0/i...,BORE-1044,32,99750.0,flat,sale,93.0,3,1,"calle Boo, 9",Cantabria,Guarnizo,NaN,es,43.406520,-3.839502,True,https://www.idealista.com/inmueble/111067361/,1503,NO COBRAMOS HONORARIOS AL COMPRADOR.. . Te pre...,False,renew,False,1073.0,True,False,False,False,[],False,False,False,99750.0,€,NaN,NaN,flat,NaN,Guarnizo,"Piso en calle Boo, 9",NaN,1,True,False,NaN,NaN,NaN,NaN,NaN
2,111065389,https://img4.idealista.com/blur/480_360_mq/0/i...,8150,17,180000.0,flat,sale,67.0,2,1,avenida del Deporte,Cantabria,Santander,Alisal - Cazoña - San Román,es,43.460021,-3.844195,False,https://www.idealista.com/inmueble/111065389/,5068,� EXCLUSIVA VIVIENDA CON GARAJE EN AVENIDA DEL...,False,good,False,2687.0,False,False,False,False,[],False,False,False,180000.0,€,NaN,NaN,flat,NaN,"Alisal - Cazoña - San Román, Santander",Piso en avenida del Deporte,Destacado,bj,True,False,NaN,NaN,NaN,NaN,NaN
3,111062244,https://img4.idealista.com/blur/480_360_mq/0/i...,GTRE-01306,42,376000.0,chalet,sale,166.0,4,3,calle Ría de Solía,Cantabria,El Astillero,NaN,es,43.392980,-3.822827,False,https://www.idealista.com/inmueble/111062244/,3512,NO COBRAMOS HONORARIOS AL COMPRADOR Te prese...,False,renew,False,2265.0,True,False,False,False,[],False,False,False,376000.0,€,True,True,chalet,terracedHouse,El Astillero,Chalet adosado en calle Ría de Solía,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,111062226,https://img4.idealista.com/blur/480_360_mq/0/i...,PL195,41,320000.0,flat,sale,120.0,3,2,calle los Plátanos,Cantabria,Santander,Alisal - Cazoña - San Román,es,43.459310,-3.850836,False,https://www.idealista.com/inmueble/111062226/,4933,"DÚPLEX EN VENTA EN SANTANDER AMPLIO, LUMINOSO ...",False,good,False,2667.0,False,False,False,False,[],False,False,False,320000.0,€,True,True,flat,NaN,"Alisal - Cazoña - San Román, Santander",Piso en calle los Plátanos,NaN,4,True,True,NaN,NaN,NaN,NaN,NaN


In [4]:
print(f"Columnas totales: {df.columns}")

columns = ['propertyCode','price','size','rooms','bathrooms','detailedType.typology','detailedType.subTypology',
                'province','municipality','district','latitude','longitude','floor','exterior','hasLift',
                'parkingSpace.hasParkingSpace','newDevelopment']

df_reduced = df[columns].copy()
df_reduced.head()

Columnas totales: Index(['propertyCode', 'thumbnail', 'externalReference', 'numPhotos', 'price', 'propertyType', 'operation', 'size', 'rooms', 'bathrooms',
       'address', 'province', 'municipality', 'district', 'country', 'latitude', 'longitude', 'showAddress', 'url', 'distance',
       'description', 'hasVideo', 'status', 'newDevelopment', 'priceByArea', 'hasPlan', 'has3DTour', 'has360', 'hasStaging', 'notes',
       'topNewDevelopment', 'newDevelopmentHighlight', 'topPlus', 'priceInfo.price.amount', 'priceInfo.price.currencySuffix',
       'parkingSpace.hasParkingSpace', 'parkingSpace.isParkingSpaceIncludedInPrice', 'detailedType.typology', 'detailedType.subTypology',
       'suggestedTexts.subtitle', 'suggestedTexts.title', 'highlight.groupDescription', 'floor', 'exterior', 'hasLift',
       'priceInfo.price.priceDropInfo.formerPrice', 'priceInfo.price.priceDropInfo.priceDropValue',
       'priceInfo.price.priceDropInfo.priceDropPercentage', 'parkingSpace.parkingSpacePrice', 'new

,propertyCode,price,size,rooms,bathrooms,detailedType.typology,detailedType.subTypology,province,municipality,district,latitude,longitude,floor,exterior,hasLift,parkingSpace.hasParkingSpace,newDevelopment
0,111067785,600000.0,216.0,5,3,chalet,semidetachedHouse,Cantabria,Santa Cruz de Bezana,Bezana,43.445929,-3.904541,NaN,NaN,NaN,True,False
1,111067361,99750.0,93.0,3,1,flat,NaN,Cantabria,Guarnizo,NaN,43.406520,-3.839502,1,True,False,NaN,False
2,111065389,180000.0,67.0,2,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.460021,-3.844195,bj,True,False,NaN,False
3,111062244,376000.0,166.0,4,3,chalet,terracedHouse,Cantabria,El Astillero,NaN,43.392980,-3.822827,NaN,NaN,NaN,True,False
4,111062226,320000.0,120.0,3,2,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.459310,-3.850836,4,True,True,True,False


In [5]:
columnas_espanol = {
    "propertyCode": "codigo_inmueble",
    "price": "precio",
    "size": "superficie_construida_m2",
    "rooms": "numero_dormitorios",
    "bathrooms": "numero_banos",
    "detailedType.typology": "tipologia",
    "detailedType.subTypology": "subtipologia",
    "operation": "tipo_operacion",
    "province": "provincia",
    "municipality": "municipio",
    "district": "distrito",
    "latitude": "latitud",
    "longitude": "longitud",
    "floor": "planta",
    "exterior": "es_exterior",
    "hasLift": "tiene_ascensor",
    "parkingSpace.hasParkingSpace": "tiene_garaje",
    "newDevelopment": "obra_nueva"
}

df_es = df_reduced.rename(columns = columnas_espanol).copy()
df_es.head(10)

,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
0,111067785,600000.0,216.0,5,3,chalet,semidetachedHouse,Cantabria,Santa Cruz de Bezana,Bezana,43.445929,-3.904541,NaN,NaN,NaN,True,False
1,111067361,99750.0,93.0,3,1,flat,NaN,Cantabria,Guarnizo,NaN,43.406520,-3.839502,1,True,False,NaN,False
2,111065389,180000.0,67.0,2,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.460021,-3.844195,bj,True,False,NaN,False
3,111062244,376000.0,166.0,4,3,chalet,terracedHouse,Cantabria,El Astillero,NaN,43.392980,-3.822827,NaN,NaN,NaN,True,False
4,111062226,320000.0,120.0,3,2,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.459310,-3.850836,4,True,True,True,False
5,110909856,270000.0,80.0,3,2,flat,NaN,Cantabria,Santander,Peñacastillo - Nuevamontaña,43.441434,-3.852004,3,True,True,True,False
6,111061719,195000.0,77.0,2,2,flat,NaN,Cantabria,Santander,Cuatro Caminos,43.458185,-3.820442,3,True,False,NaN,False
7,111061589,295000.0,92.0,3,1,flat,NaN,Cantabria,Santander,Castilla - Hermida,43.457346,-3.810930,6,True,True,True,False
8,111060096,109000.0,80.0,2,1,flat,NaN,Cantabria,Santander,Alisal - Cazoña - San Román,43.459158,-3.845045,1,True,True,NaN,False
9,111057640,220000.0,111.0,3,1,flat,NaN,Cantabria,Santander,Castilla - Hermida,43.455968,-3.809975,4,True,False,NaN,False


In [6]:
print(f"Total elementos duplicados: {df_es['codigo_inmueble'].duplicated().sum()} de {len(df_es)}")
df_es[df_es['codigo_inmueble'].duplicated(keep=False)].sort_values("codigo_inmueble")


Total elementos duplicados: 2078 de 4547


,codigo_inmueble,precio,superficie_construida_m2,numero_dormitorios,numero_banos,tipologia,subtipologia,provincia,municipio,distrito,latitud,longitud,planta,es_exterior,tiene_ascensor,tiene_garaje,obra_nueva
3945,32275914,735000.0,400.0,4,6,chalet,independantHouse,Cantabria,Marina de Cudeyo,NaN,43.410230,-3.780932,NaN,NaN,NaN,True,False
4196,32275914,735000.0,400.0,4,6,chalet,independantHouse,Cantabria,Marina de Cudeyo,NaN,43.410230,-3.780932,NaN,NaN,NaN,True,False
1048,36010405,550000.0,143.0,4,3,flat,duplex,Cantabria,Santander,Cuatro Caminos,43.459587,-3.833186,3,True,True,True,False
2400,36010405,550000.0,143.0,4,3,flat,duplex,Cantabria,Santander,Cuatro Caminos,43.459587,-3.833186,3,True,True,True,False
3002,81606165,1080000.0,449.0,4,5,chalet,independantHouse,Cantabria,Santa Cruz de Bezana,Soto de la Marina,43.463408,-3.899782,NaN,NaN,NaN,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,111067785,600000.0,216.0,5,3,chalet,semidetachedHouse,Cantabria,Santa Cruz de Bezana,Bezana,43.445929,-3.904541,NaN,NaN,NaN,True,False
0,111067785,600000.0,216.0,5,3,chalet,semidetachedHouse,Cantabria,Santa Cruz de Bezana,Bezana,43.445929,-3.904541,NaN,NaN,NaN,True,False
450,111070179,595000.0,349.0,5,3,chalet,independantHouse,Cantabria,Cudon,NaN,43.411125,-4.013922,NaN,NaN,NaN,True,False
250,111070179,595000.0,349.0,5,3,chalet,independantHouse,Cantabria,Cudon,NaN,43.411125,-4.013922,NaN,NaN,NaN,True,False


In [7]:
df_clean = df_es.drop_duplicates(subset="codigo_inmueble", keep="first").copy()
print(f"Forma del nuevo dataset: {df_clean.shape}")


Forma del nuevo dataset: (2469, 17)


In [8]:
df_clean['municipio'].value_counts().head(10)


municipio
Santander               783
Castro-Urdiales         376
Laredo                  175
Piélagos                151
Camargo                 118
Santa Cruz de Bezana    106
Suances                 104
Santoña                  95
Ribamontan al Mar        73
Barcena de Cicero        64
Name: count, dtype: int64

In [9]:
df_expandido = agregar_distancias_minimas_poi(df_clean, ["playa", "supermercado", "colegio"])


In [10]:
# Export CSV limpio
OUTPUT_DIR = Path("../../data/processed/idealistaAPI")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "sale_homes_clean.csv"
df_expandido.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV limpio exportado en: {OUTPUT_CSV.resolve()}")


CSV limpio exportado en: /Users/sitomachucas/Documents/BezanillaSL/data/processed/idealistaAPI/sale_homes_clean.csv
